In [21]:
import pandas as pd
import numpy as np

# Path mapeado para o ambiente local
caminho_arquivo = "/home/matheus/Documents/CASE SANOVA/micromedicao/data/raw/Dados - Estudo Micromedição.xlsx"

df_raw = pd.read_excel(caminho_arquivo)

print("Diagnóstico inicial da base:")
print(f"Total de Linhas: {df_raw.shape[0]} | Total de Colunas: {df_raw.shape[1]}\n")

Diagnóstico inicial da base:
Total de Linhas: 1912 | Total de Colunas: 132



In [22]:
# Padronização de tipos e formatos
# As datas seguem o padrão nacional (DD/MM/AAAA) e colunas de métricas devem ser numéricas

df = df_raw.copy()

# Normalização das colunas de texto (remoção de espaços e capitalização)
str_cols = [
    'SIT._LIG_AGUA', 'SIT._LIG_ESGOTO', 'TIPO_HIDROMETRO',
    'MARCA_HIDROMETRO', 'CLASSE_METROLOGICA', 'CATEGORIA_PRINCIPAL',
    'DIAMETRO_HIDROMETRO'
]
for col in str_cols:
    df[col] = df[col].astype(str).str.strip().str.title()
    df[col] = df[col].replace('Nan', np.nan)

df['DATA_INSTALACAO_HIDROMETRO'] = pd.to_datetime(
    df['DATA_INSTALACAO_HIDROMETRO'],
    dayfirst=True,
    errors='coerce'
)

volume_valor_cols = [col for col in df.columns if any(
    col.startswith(p) for p in ['VOLUME_', 'VALOR_', 'NUMERO_ECONOMIAS']
)]
for col in volume_valor_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("Tipos de dados após padronização (amostra):\n")
print(df[['SIT._LIG_AGUA', 'DATA_INSTALACAO_HIDROMETRO', 'VOLUME_LIDO', 'VALOR_TOTAL']].dtypes)
print(f"\nDatas nulas após conversão: {df['DATA_INSTALACAO_HIDROMETRO'].isna().sum()}")

Tipos de dados após padronização (amostra):

SIT._LIG_AGUA                            str
DATA_INSTALACAO_HIDROMETRO    datetime64[us]
VOLUME_LIDO                          float64
VALOR_TOTAL                          float64
dtype: object

Datas nulas após conversão: 82


In [23]:
# Diagnóstico de campos nulos
# Nulos em campos de hidrômetro são normais em ligações inativas.
# Em ligações ativas, nulos em métricas de volume/valor indicam ausência de faturamento no período.

nulos = df.isnull().sum()
nulos_relevantes = nulos[nulos > 0].sort_values(ascending=False)
print("Mapeamento de valores nulos na base:\n")
print(nulos_relevantes)

# Segmentação por status da ligação de água
STATUS_ATIVA = 'Ativa'

df_ativas = df[df['SIT._LIG_AGUA'] == STATUS_ATIVA].copy()
df_nao_ativas = df[df['SIT._LIG_AGUA'] != STATUS_ATIVA].copy()

print(f"\nLigações Ativas: {len(df_ativas)}")
print(f"Ligações Não Ativas: {len(df_nao_ativas)}")

nulos_ativas = df_ativas.isnull().sum()
nulos_ativas_relevantes = nulos_ativas[nulos_ativas > 0].sort_values(ascending=False)
print("\nDetalhamento de nulos apenas nas ligações Ativas:\n")
print(nulos_ativas_relevantes)

# Identificação de falha cadastral: ligação ativa sem hidrômetro vinculado
df_ativas['FLAG_SEM_HIDROMETRO'] = df_ativas['NUMERO_HIDROMETRO'].isna().astype(int)
sem_hidrometro_ativas = df_ativas['FLAG_SEM_HIDROMETRO'].sum()
print(f"\nAlerta: Ligações ativas sem hidrômetro cadastrado: {sem_hidrometro_ativas}")

Mapeamento de valores nulos na base:

NUMERO_ECONOMIAS_PUB          1907
NUMERO_ECONOMIAS_IND          1829
NUMERO_ECONOMIAS_COM          1755
NUMERO_ECONOMIAS_RES           247
VALOR_DESCONTOS_12             177
                              ... 
MARCA_HIDROMETRO                82
DATA_INSTALACAO_HIDROMETRO      82
SIT._LIG_ESGOTO                 36
CATEGORIA_PRINCIPAL             17
SIT._LIG_AGUA                   15
Length: 131, dtype: int64

Ligações Ativas: 1815
Ligações Não Ativas: 97

Detalhamento de nulos apenas nas ligações Ativas:

NUMERO_ECONOMIAS_PUB     1811
NUMERO_ECONOMIAS_IND     1749
NUMERO_ECONOMIAS_COM     1673
NUMERO_ECONOMIAS_RES      203
VALOR_DESCONTOS_12        106
                         ... 
CLASSE_METROLOGICA          1
TIPO_HIDROMETRO             1
MARCA_HIDROMETRO            1
DIAMETRO_HIDROMETRO         1
CAPACIDADE_HIDROMETRO       1
Length: 130, dtype: int64

Alerta: Ligações ativas sem hidrômetro cadastrado: 1


In [24]:
# Consolidação das flags cadastrais iniciais

economias_cols = ['NUMERO_ECONOMIAS_RES','NUMERO_ECONOMIAS_COM',
                  'NUMERO_ECONOMIAS_IND','NUMERO_ECONOMIAS_PUB']

flag_sit_agua_nula = df['SIT._LIG_AGUA'].isna().astype(int)
flag_sit_esgoto_nula = df['SIT._LIG_ESGOTO'].isna().astype(int)
flag_sem_categoria = df['CATEGORIA_PRINCIPAL'].isna().astype(int)
flag_sem_hidrometro = (
    (df['SIT._LIG_AGUA'] == 'Ativa') &
    (df['NUMERO_HIDROMETRO'].isna())
).astype(int)

df_ativas_temp = df[df['SIT._LIG_AGUA'] == 'Ativa']
sem_economia_idx = df_ativas_temp[
    df_ativas_temp[economias_cols].sum(axis=1) == 0
].index

flag_sem_economia = pd.Series(0, index=df.index)
flag_sem_economia.loc[sem_economia_idx] = 1

flags_df = pd.DataFrame({
    'FLAG_SIT_AGUA_NULA': flag_sit_agua_nula,
    'FLAG_SIT_ESGOTO_NULA': flag_sit_esgoto_nula,
    'FLAG_SEM_CATEGORIA': flag_sem_categoria,
    'FLAG_SEM_HIDROMETRO': flag_sem_hidrometro,
    'FLAG_SEM_ECONOMIA': flag_sem_economia,
}, index=df.index)

# Adicionando ao dataframe principal
cols_flags_antigas = [c for c in df.columns if c.startswith('FLAG_')]
df = df.drop(columns=cols_flags_antigas)
df = pd.concat([df, flags_df], axis=1).copy()

print("Resumo das flags cadastrais:\n")
for flag in flags_df.columns:
    total = df[flag].sum()
    print(f"{flag}: {total} ligações")

print(f"\nShape atualizado do DataFrame: {df.shape}")

Resumo das flags cadastrais:

FLAG_SIT_AGUA_NULA: 15 ligações
FLAG_SIT_ESGOTO_NULA: 36 ligações
FLAG_SEM_CATEGORIA: 17 ligações
FLAG_SEM_HIDROMETRO: 1 ligações
FLAG_SEM_ECONOMIA: 6 ligações

Shape atualizado do DataFrame: (1912, 137)


In [25]:
# Tratamento de códigos de ocorrência nos volumes lidos
# Valores de VOLUME_LIDO a partir de 900.000 representam ocorrências de leitura
# (ex: hidrômetro parado, sem acesso). Substituímos por NaN para não distorcer as análises,
# mantendo o VOLUME_REAL como referência de consumo validado.

volume_lido_cols = [c for c in df.columns if 'VOLUME_LIDO' in c]
THRESHOLD_OCORRENCIA = 900_000

print("Diagnóstico de volumes suspeitos (>= 900.000 m³):\n")
total_absurdos = 0
for col in volume_lido_cols:
    n = (df[col] >= THRESHOLD_OCORRENCIA).sum()
    if n > 0:
        print(f"  {col}: {n} ocorrências")
        total_absurdos += n
print(f"\nTotal de leituras tratadas como ocorrência: {total_absurdos}")

# Flag para manter o histórico das ligações com ocorrência
df['FLAG_LEITURA_OCORRENCIA'] = (
    df[volume_lido_cols].ge(THRESHOLD_OCORRENCIA).any(axis=1)
).astype(int)

print(f"\nLigações impactadas com pelo menos 1 ocorrência: {df['FLAG_LEITURA_OCORRENCIA'].sum()}")

# Substituição dos outliers
for col in volume_lido_cols:
    mask = df[col] >= THRESHOLD_OCORRENCIA
    df.loc[mask, col] = np.nan

print("\nValores discrepantes substituídos por NaN nas colunas VOLUME_LIDO.")
print("VOLUME_REAL mantido intacto como referência de consumo efetivo.")

restantes = sum((df[col] >= THRESHOLD_OCORRENCIA).sum() for col in volume_lido_cols)
print(f"\nValores restantes acima do limite após tratamento: {restantes}")

Diagnóstico de volumes suspeitos (>= 900.000 m³):

  VOLUME_LIDO: 10 ocorrências
  VOLUME_LIDO_01: 4 ocorrências
  VOLUME_LIDO_02: 12 ocorrências
  VOLUME_LIDO_03: 4 ocorrências
  VOLUME_LIDO_04: 5 ocorrências
  VOLUME_LIDO_05: 5 ocorrências
  VOLUME_LIDO_06: 5 ocorrências
  VOLUME_LIDO_07: 5 ocorrências
  VOLUME_LIDO_08: 3 ocorrências
  VOLUME_LIDO_09: 8 ocorrências
  VOLUME_LIDO_10: 35 ocorrências
  VOLUME_LIDO_11: 3 ocorrências
  VOLUME_LIDO_12: 8 ocorrências

Total de leituras tratadas como ocorrência: 107

Ligações impactadas com pelo menos 1 ocorrência: 87

Valores discrepantes substituídos por NaN nas colunas VOLUME_LIDO.
VOLUME_REAL mantido intacto como referência de consumo efetivo.

Valores restantes acima do limite após tratamento: 0


In [26]:
# Identificação de inconsistências financeiras e de situação

# Ligação com água inativa, mas esgoto ativo indica falha na sincronia do cadastro
df['FLAG_INCONSISTENCIA_SITUACAO'] = (
    (df['SIT._LIG_AGUA'] != 'Ativa') &
    (df['SIT._LIG_ESGOTO'] == 'Ativa')
).astype(int)

print("--- Inconsistência de Situação ---")
print(f"Ligações com água inativa e esgoto ativo: {df['FLAG_INCONSISTENCIA_SITUACAO'].sum()}")

# Valores negativos indicam estornos sistêmicos aplicados na fatura
valor_servicos_cols = [c for c in df.columns if 'VALOR_SERVICOS' in c]

mask_neg_servicos = df[valor_servicos_cols].lt(0).any(axis=1)
df['FLAG_SERVICO_NEGATIVO'] = mask_neg_servicos.astype(int)

print(f"\n--- Serviços Negativos (Estornos) ---")
print(f"Ligações com valor de serviço negativo em algum mês: {df['FLAG_SERVICO_NEGATIVO'].sum()}\n")

print("Distribuição das ocorrências por período:")
for col in valor_servicos_cols:
    n = (df[col] < 0).sum()
    if n > 0:
        print(f"  {col}: {n} ocorrências | Menor valor registrado: {df[col].min():.2f}")

# Identificação de possível perda de receita ou erro no cálculo da fatura
mask_ativa_zero = (
    (df['SIT._LIG_AGUA'] == 'Ativa') &
    (df['VALOR_TOTAL'] == 0) &
    (df['VOLUME_FATURADO'] > 0)
)
df['FLAG_FATURAMENTO_ZERADO'] = mask_ativa_zero.astype(int)

print(f"\n--- Faturamento Zerado ---")
print(f"Ligações ativas com consumo faturado mas sem cobrança no mês base: {df['FLAG_FATURAMENTO_ZERADO'].sum()}\n")
print("Detalhamento dos casos:")
print(df[mask_ativa_zero][['MATRICULA','CATEGORIA_PRINCIPAL',
      'VOLUME_FATURADO','VALOR_SERVICOS','VALOR_TOTAL']].to_string())

df = df.copy()
print(f"\nShape atual do DataFrame: {df.shape}")

--- Inconsistência de Situação ---
Ligações com água inativa e esgoto ativo: 65

--- Serviços Negativos (Estornos) ---
Ligações com valor de serviço negativo em algum mês: 128

Distribuição das ocorrências por período:
  VALOR_SERVICOS: 10 ocorrências | Menor valor registrado: -7019.68
  VALOR_SERVICOS_01: 19 ocorrências | Menor valor registrado: -4707.72
  VALOR_SERVICOS_02: 21 ocorrências | Menor valor registrado: -4843.82
  VALOR_SERVICOS_03: 32 ocorrências | Menor valor registrado: -4482.22
  VALOR_SERVICOS_04: 23 ocorrências | Menor valor registrado: -5640.95
  VALOR_SERVICOS_05: 21 ocorrências | Menor valor registrado: -7176.31
  VALOR_SERVICOS_06: 29 ocorrências | Menor valor registrado: -6006.44
  VALOR_SERVICOS_07: 31 ocorrências | Menor valor registrado: -5711.69
  VALOR_SERVICOS_08: 31 ocorrências | Menor valor registrado: -280240.43
  VALOR_SERVICOS_09: 24 ocorrências | Menor valor registrado: -8436.93
  VALOR_SERVICOS_10: 15 ocorrências | Menor valor registrado: -13505.58


In [27]:
# Investigação detalhada do estorno extremo em Agosto e quebra do faturamento zerado

outlier_ago = df[df['VALOR_SERVICOS_08'] == df['VALOR_SERVICOS_08'].min()]
print("--- Investigação de Outlier: Estorno Extremo (Agosto) ---\n")
print(outlier_ago[['MATRICULA', 'CATEGORIA_PRINCIPAL',
                    'NUMERO_ECONOMIAS_RES', 'NUMERO_ECONOMIAS_COM',
                    'NUMERO_ECONOMIAS_IND', 'VOLUME_FATURADO_08',
                    'VALOR_AGUA_08', 'VALOR_ESGOTO_08',
                    'VALOR_SERVICOS_08', 'VALOR_TOTAL_08']].to_string())

# Classificação do faturamento zerado para dimensionar o impacto financeiro
mask_ativa_zero = (
    (df['SIT._LIG_AGUA'] == 'Ativa') &
    (df['VALOR_TOTAL'] == 0) &
    (df['VOLUME_FATURADO'] > 0)
)

zerado_com_estorno = df[mask_ativa_zero & (df['VALOR_SERVICOS'] < 0)]
zerado_sem_estorno = df[mask_ativa_zero & (df['VALOR_SERVICOS'] >= 0)]

print(f"\n--- Classificação do Faturamento Zerado ---\n")
print(f"Tipo 1: Zerado por aplicação de estorno: {len(zerado_com_estorno)} ligações")
receita_estorno = (zerado_com_estorno['VALOR_AGUA'] + zerado_com_estorno['VALOR_ESGOTO']).sum()
print(f"Potencial bloqueado (Água + Esgoto): R$ {receita_estorno:,.2f}\n")

print(f"Tipo 2: Zerado sem justificativa de estorno (possível erro): {len(zerado_sem_estorno)} ligações")
receita_erro = (zerado_sem_estorno['VALOR_AGUA'] + zerado_sem_estorno['VALOR_ESGOTO']).sum()
print(f"Receita em risco (Água + Esgoto): R$ {receita_erro:,.2f}\n")

print(f"Impacto financeiro total mapeado no mês base: R$ {(receita_estorno + receita_erro):,.2f}")

# Flag específica para estornos excepcionais
THRESHOLD_SERVICO_EXTREMO = -50_000
df['FLAG_SERVICO_EXTREMO'] = (
    df[valor_servicos_cols].lt(THRESHOLD_SERVICO_EXTREMO).any(axis=1)
).astype(int)

print(f"\nFlag para estornos extremos (<= {THRESHOLD_SERVICO_EXTREMO}): {df['FLAG_SERVICO_EXTREMO'].sum()} ocorrências registradas.")

df = df.copy()
print(f"\nShape atual do DataFrame: {df.shape}")

--- Investigação de Outlier: Estorno Extremo (Agosto) ---

     MATRICULA CATEGORIA_PRINCIPAL  NUMERO_ECONOMIAS_RES  NUMERO_ECONOMIAS_COM  NUMERO_ECONOMIAS_IND  VOLUME_FATURADO_08  VALOR_AGUA_08  VALOR_ESGOTO_08  VALOR_SERVICOS_08  VALOR_TOTAL_08
416  1371190-3         Residencial                   1.0                   NaN                   NaN              9982.0      155838.69        124670.95         -280240.43          269.21

--- Classificação do Faturamento Zerado ---

Tipo 1: Zerado por aplicação de estorno: 9 ligações
Potencial bloqueado (Água + Esgoto): R$ 15,022.62

Tipo 2: Zerado sem justificativa de estorno (possível erro): 8 ligações
Receita em risco (Água + Esgoto): R$ 1,878.71

Impacto financeiro total mapeado no mês base: R$ 16,901.33

Flag para estornos extremos (<= -50000): 1 ocorrências registradas.

Shape atual do DataFrame: (1912, 142)


In [28]:
# Geração de features baseadas no comportamento histórico (últimos 12 meses)
# O mês atual (base) não compõe o histórico, sendo usado como referência para desvios.

volume_real_cols  = [f'VOLUME_REAL_{str(i).zfill(2)}'    for i in range(1,13)]
volume_fat_cols   = [f'VOLUME_FATURADO_{str(i).zfill(2)}' for i in range(1,13)]
valor_total_cols  = [f'VALOR_TOTAL_{str(i).zfill(2)}'    for i in range(1,13)]
valor_agua_cols   = [f'VALOR_AGUA_{str(i).zfill(2)}'     for i in range(1,13)]
valor_esgoto_cols = [f'VALOR_ESGOTO_{str(i).zfill(2)}'   for i in range(1,13)]

cols_aux_antigas = [
    'MEDIA_CONSUMO_HISTORICO','TOTAL_CONSUMO_HISTORICO',
    'MAX_CONSUMO_HISTORICO','MIN_CONSUMO_HISTORICO',
    'STD_CONSUMO_HISTORICO','TOTAL_FATURADO_HISTORICO',
    'MEDIA_FATURADO_HISTORICO','TOTAL_AGUA_HISTORICO',
    'TOTAL_ESGOTO_HISTORICO','MESES_CONSUMO_ZERO',
    'DIFF_REAL_FATURADO_BASE','FLAG_CONSUMO_ZERO_RECORRENTE',
    'CV_CONSUMO','FLAG_ALTA_VARIABILIDADE',
    'DESVIO_CONSUMO_ATUAL','DESVIO_PCT_CONSUMO_ATUAL'
]
df = df.drop(columns=[c for c in cols_aux_antigas if c in df.columns])

df = df.assign(
    MEDIA_CONSUMO_HISTORICO  = df[volume_real_cols].mean(axis=1),
    TOTAL_CONSUMO_HISTORICO  = df[volume_real_cols].sum(axis=1),
    MAX_CONSUMO_HISTORICO    = df[volume_real_cols].max(axis=1),
    MIN_CONSUMO_HISTORICO    = df[volume_real_cols].min(axis=1),
    STD_CONSUMO_HISTORICO    = df[volume_real_cols].std(axis=1),
    TOTAL_FATURADO_HISTORICO = df[valor_total_cols].sum(axis=1),
    MEDIA_FATURADO_HISTORICO = df[valor_total_cols].mean(axis=1),
    TOTAL_AGUA_HISTORICO     = df[valor_agua_cols].sum(axis=1),
    TOTAL_ESGOTO_HISTORICO   = df[valor_esgoto_cols].sum(axis=1),
    MESES_CONSUMO_ZERO       = (df[volume_real_cols] == 0).sum(axis=1),
    DIFF_REAL_FATURADO_BASE  = df['VOLUME_FATURADO'] - df['VOLUME_REAL'],
)

df = df.assign(
    FLAG_CONSUMO_ZERO_RECORRENTE = (df['MESES_CONSUMO_ZERO'] > 3).astype(int),
    CV_CONSUMO = df['STD_CONSUMO_HISTORICO'] / df['MEDIA_CONSUMO_HISTORICO'].replace(0, np.nan)
).assign(
    FLAG_ALTA_VARIABILIDADE = lambda x: (x['CV_CONSUMO'] > 1.5).astype(int),
    DESVIO_CONSUMO_ATUAL    = lambda x: df['VOLUME_REAL'] - x['MEDIA_CONSUMO_HISTORICO'],
).assign(
    DESVIO_PCT_CONSUMO_ATUAL = lambda x: (x['DESVIO_CONSUMO_ATUAL'] / x['MEDIA_CONSUMO_HISTORICO'].replace(0, np.nan) * 100).round(2)
)

df = df.copy()

print("Resumo estatístico das features históricas criadas:\n")
aux_cols = [
    'MEDIA_CONSUMO_HISTORICO','TOTAL_CONSUMO_HISTORICO',
    'MAX_CONSUMO_HISTORICO','MIN_CONSUMO_HISTORICO',
    'STD_CONSUMO_HISTORICO','TOTAL_FATURADO_HISTORICO',
    'MEDIA_FATURADO_HISTORICO','MESES_CONSUMO_ZERO',
    'DIFF_REAL_FATURADO_BASE','CV_CONSUMO',
    'DESVIO_CONSUMO_ATUAL','DESVIO_PCT_CONSUMO_ATUAL'
]
print(df[aux_cols].describe().round(2).to_string())

print("\n--- Detecção de Comportamento Anômalo ---")
print(f"Consumo zerado em mais de 3 meses do histórico: {df['FLAG_CONSUMO_ZERO_RECORRENTE'].sum()} ligações")
print(f"Variabilidade de consumo atípica (CV > 1.5): {df['FLAG_ALTA_VARIABILIDADE'].sum()} ligações")

print(f"\nShape atual do DataFrame: {df.shape}")

Resumo estatístico das features históricas criadas:

       MEDIA_CONSUMO_HISTORICO  TOTAL_CONSUMO_HISTORICO  MAX_CONSUMO_HISTORICO  MIN_CONSUMO_HISTORICO  STD_CONSUMO_HISTORICO  TOTAL_FATURADO_HISTORICO  MEDIA_FATURADO_HISTORICO  MESES_CONSUMO_ZERO  DIFF_REAL_FATURADO_BASE  CV_CONSUMO  DESVIO_CONSUMO_ATUAL  DESVIO_PCT_CONSUMO_ATUAL
count                  1824.00                  1912.00                1824.00                1824.00                1816.00                   1912.00                   1822.00             1912.00                  1829.00     1800.00               1817.00                   1801.00
mean                     35.71                   406.59                  62.53                  21.71                  12.67                   5940.30                    526.90                0.63                     5.13        0.48                  5.07                     38.39
std                     121.81                  1430.93                 281.59                  97.10

In [29]:
# Recriação consolidada das flags para garantir integridade estrutural antes do scoring

valor_servicos_cols = [c for c in df.columns if 'VALOR_SERVICOS' in c]
economias_cols = ['NUMERO_ECONOMIAS_RES','NUMERO_ECONOMIAS_COM',
                  'NUMERO_ECONOMIAS_IND','NUMERO_ECONOMIAS_PUB']

flags_recriadas = pd.DataFrame(index=df.index)

flags_recriadas['FLAG_SIT_AGUA_NULA']   = df['SIT._LIG_AGUA'].isna().astype(int)
flags_recriadas['FLAG_SIT_ESGOTO_NULA'] = df['SIT._LIG_ESGOTO'].isna().astype(int)
flags_recriadas['FLAG_SEM_CATEGORIA']   = df['CATEGORIA_PRINCIPAL'].isna().astype(int)
flags_recriadas['FLAG_SEM_HIDROMETRO']  = ((df['SIT._LIG_AGUA'] == 'Ativa') & (df['NUMERO_HIDROMETRO'].isna())).astype(int)

sem_economia_idx = df[(df['SIT._LIG_AGUA'] == 'Ativa') & (df[economias_cols].sum(axis=1) == 0)].index
flags_recriadas['FLAG_SEM_ECONOMIA'] = 0
flags_recriadas.loc[sem_economia_idx, 'FLAG_SEM_ECONOMIA'] = 1

flags_recriadas['FLAG_INCONSISTENCIA_SITUACAO'] = ((df['SIT._LIG_AGUA'] != 'Ativa') & (df['SIT._LIG_ESGOTO'] == 'Ativa')).astype(int)
flags_recriadas['FLAG_SERVICO_NEGATIVO'] = df[valor_servicos_cols].lt(0).any(axis=1).astype(int)
flags_recriadas['FLAG_FATURAMENTO_ZERADO'] = ((df['SIT._LIG_AGUA'] == 'Ativa') & (df['VALOR_TOTAL'] == 0) & (df['VOLUME_FATURADO'] > 0)).astype(int)
flags_recriadas['FLAG_SERVICO_EXTREMO'] = df[valor_servicos_cols].lt(-50_000).any(axis=1).astype(int)

flags_cols_antigas = [c for c in df.columns if c.startswith('FLAG_')]
df = df.drop(columns=flags_cols_antigas)
df = pd.concat([df, flags_recriadas], axis=1).copy()

print("Auditoria final das flags consolidadas:\n")
for f in flags_recriadas.columns:
    print(f"  {f}: {df[f].sum()}")
    
print(f"\nTotal de regras mapeadas: {len(flags_recriadas.columns)}")
print(f"Shape atual do DataFrame: {df.shape}")

Auditoria final das flags consolidadas:

  FLAG_SIT_AGUA_NULA: 15
  FLAG_SIT_ESGOTO_NULA: 36
  FLAG_SEM_CATEGORIA: 17
  FLAG_SEM_HIDROMETRO: 1
  FLAG_SEM_ECONOMIA: 6
  FLAG_INCONSISTENCIA_SITUACAO: 65
  FLAG_SERVICO_NEGATIVO: 128
  FLAG_FATURAMENTO_ZERADO: 17
  FLAG_SERVICO_EXTREMO: 1

Total de regras mapeadas: 9
Shape atual do DataFrame: (1912, 155)


In [30]:
# Definição do Score de Risco para priorização das análises de campo ou auditoria comercial

df['FLAG_TODOS_MESES_ZERO'] = (
    (df['SIT._LIG_AGUA'] == 'Ativa') & 
    (df['MESES_CONSUMO_ZERO'] == 12)
).astype(int)

df['FLAG_SUBFATURAMENTO'] = (df['DIFF_REAL_FATURADO_BASE'] < 0).astype(int)

data_corte_antigo = pd.Timestamp('2016-05-14')
df['FLAG_HIDROMETRO_ANTIGO'] = (
    (df['DATA_INSTALACAO_HIDROMETRO'] < data_corte_antigo) & 
    (df['SIT._LIG_AGUA'] == 'Ativa')
).astype(int)

# Cálculo do Score de Risco (soma das flags ativadas)
flag_cols = [c for c in df.columns if c.startswith('FLAG_')]
df['SCORE_RISCO'] = df[flag_cols].sum(axis=1)

df = df.copy()

print("Mapeamento da representatividade de cada flag na base:\n")
for flag in flag_cols:
    total = df[flag].sum()
    pct = total / len(df) * 100
    print(f"  {flag:<40} {total:>5} ligações  ({pct:.1f}%)")

print("\nDistribuição do Score de Risco:\n")
print(df['SCORE_RISCO'].value_counts().sort_index().to_string())

print(f"\nResumo de criticidade:")
print(f"Alto Risco (score >= 3): {(df['SCORE_RISCO'] >= 3).sum()} casos")
print(f"Risco Médio (score >= 2): {(df['SCORE_RISCO'] >= 2).sum()} casos")
print(f"Sem anomalias identificadas (score = 0): {(df['SCORE_RISCO'] == 0).sum()} casos")

print(f"\nShape final do DataFrame: {df.shape}")

Mapeamento da representatividade de cada flag na base:

  FLAG_SIT_AGUA_NULA                          15 ligações  (0.8%)
  FLAG_SIT_ESGOTO_NULA                        36 ligações  (1.9%)
  FLAG_SEM_CATEGORIA                          17 ligações  (0.9%)
  FLAG_SEM_HIDROMETRO                          1 ligações  (0.1%)
  FLAG_SEM_ECONOMIA                            6 ligações  (0.3%)
  FLAG_INCONSISTENCIA_SITUACAO                65 ligações  (3.4%)
  FLAG_SERVICO_NEGATIVO                      128 ligações  (6.7%)
  FLAG_FATURAMENTO_ZERADO                     17 ligações  (0.9%)
  FLAG_SERVICO_EXTREMO                         1 ligações  (0.1%)
  FLAG_TODOS_MESES_ZERO                       10 ligações  (0.5%)
  FLAG_SUBFATURAMENTO                          0 ligações  (0.0%)
  FLAG_HIDROMETRO_ANTIGO                       0 ligações  (0.0%)

Distribuição do Score de Risco:

SCORE_RISCO
0    1668
1     210
2      16
3      18

Resumo de criticidade:
Alto Risco (score >= 3): 18 casos
Risco Mé

In [31]:
# Reincluir flags comportamentais que foram perdidas na consolidação
df['FLAG_CONSUMO_ZERO_RECORRENTE'] = (df['MESES_CONSUMO_ZERO'] > 3).astype(int)
df['FLAG_ALTA_VARIABILIDADE']      = (df['CV_CONSUMO'] > 1.5).astype(int)

# Recalcular o SCORE_RISCO com todas as 14 flags
flag_cols = [c for c in df.columns if c.startswith('FLAG_')]
df['SCORE_RISCO'] = df[flag_cols].sum(axis=1)
df = df.copy()

print(f"Total de flags: {len(flag_cols)}")
print(f"\nConsumo zero recorrente: {df['FLAG_CONSUMO_ZERO_RECORRENTE'].sum()}")
print(f"Alta variabilidade:      {df['FLAG_ALTA_VARIABILIDADE'].sum()}")

print(f"\nDistribuicao do score:")
print(df['SCORE_RISCO'].value_counts().sort_index())

print(f"\nAlto risco (>= 3):  {(df['SCORE_RISCO'] >= 3).sum()}")
print(f"Risco medio (>= 2): {(df['SCORE_RISCO'] >= 2).sum()}")
print(f"Sem anomalia (= 0): {(df['SCORE_RISCO'] == 0).sum()}")

Total de flags: 14

Consumo zero recorrente: 124
Alta variabilidade:      89

Distribuicao do score:
SCORE_RISCO
0    1552
1     240
2      93
3      25
4       2
Name: count, dtype: int64

Alto risco (>= 3):  27
Risco medio (>= 2): 120
Sem anomalia (= 0): 1552


In [32]:
import os

df_final_ativas     = df[df['SIT._LIG_AGUA'] == 'Ativa'].copy()
df_final_nao_ativas = df[df['SIT._LIG_AGUA'] != 'Ativa'].copy()

output_path = "/home/matheus/Documents/CASE SANOVA/micromedicao/data/processed/"
os.makedirs(output_path, exist_ok=True)

df.to_excel(output_path + "base_completa_tratada.xlsx", index=False)
df_final_ativas.to_excel(output_path + "base_ativas_tratada.xlsx", index=False)
df_final_nao_ativas.to_excel(output_path + "base_nao_ativas_tratada.xlsx", index=False)

print(f"base_completa_tratada.xlsx   -> {len(df)} registros | {df.shape[1]} colunas")
print(f"base_ativas_tratada.xlsx     -> {len(df_final_ativas)} registros")
print(f"base_nao_ativas_tratada.xlsx -> {len(df_final_nao_ativas)} registros")

base_completa_tratada.xlsx   -> 1912 registros | 161 colunas
base_ativas_tratada.xlsx     -> 1815 registros
base_nao_ativas_tratada.xlsx -> 97 registros


In [33]:
# Verificação geral do estado final do df
print(f"Shape: {df.shape}")
print(f"\nFlags e scores:")
flag_cols = [c for c in df.columns if c.startswith('FLAG_')]
for f in flag_cols:
    print(f"  {f}: {df[f].sum()}")

print(f"\nScore de risco:")
print(df['SCORE_RISCO'].value_counts().sort_index())

print(f"\nAlto risco (>= 3):  {(df['SCORE_RISCO'] >= 3).sum()}")
print(f"Risco medio (>= 2): {(df['SCORE_RISCO'] >= 2).sum()}")
print(f"Sem anomalia (= 0): {(df['SCORE_RISCO'] == 0).sum()}")

Shape: (1912, 161)

Flags e scores:
  FLAG_SIT_AGUA_NULA: 15
  FLAG_SIT_ESGOTO_NULA: 36
  FLAG_SEM_CATEGORIA: 17
  FLAG_SEM_HIDROMETRO: 1
  FLAG_SEM_ECONOMIA: 6
  FLAG_INCONSISTENCIA_SITUACAO: 65
  FLAG_SERVICO_NEGATIVO: 128
  FLAG_FATURAMENTO_ZERADO: 17
  FLAG_SERVICO_EXTREMO: 1
  FLAG_TODOS_MESES_ZERO: 10
  FLAG_SUBFATURAMENTO: 0
  FLAG_HIDROMETRO_ANTIGO: 0
  FLAG_CONSUMO_ZERO_RECORRENTE: 124
  FLAG_ALTA_VARIABILIDADE: 89

Score de risco:
SCORE_RISCO
0    1552
1     240
2      93
3      25
4       2
Name: count, dtype: int64

Alto risco (>= 3):  27
Risco medio (>= 2): 120
Sem anomalia (= 0): 1552


In [37]:
# Verificação de integridade — confirma que todas as flags esperadas estão presentes
flags_esperadas = [
    'FLAG_SIT_AGUA_NULA', 'FLAG_SIT_ESGOTO_NULA', 'FLAG_SEM_CATEGORIA',
    'FLAG_SEM_HIDROMETRO', 'FLAG_SEM_ECONOMIA', 'FLAG_INCONSISTENCIA_SITUACAO',
    'FLAG_SERVICO_NEGATIVO', 'FLAG_FATURAMENTO_ZERADO', 'FLAG_SERVICO_EXTREMO',
    'FLAG_TODOS_MESES_ZERO', 'FLAG_SUBFATURAMENTO', 'FLAG_HIDROMETRO_ANTIGO',
    'FLAG_CONSUMO_ZERO_RECORRENTE', 'FLAG_ALTA_VARIABILIDADE'
]

flags_presentes  = [f for f in flags_esperadas if f in df.columns]
flags_ausentes   = [f for f in flags_esperadas if f not in df.columns]

print(f"Flags esperadas:  {len(flags_esperadas)}")
print(f"Flags presentes:  {len(flags_presentes)}")
print(f"Flags ausentes:   {len(flags_ausentes)}")

if flags_ausentes:
    print(f"\nATENCAO — flags faltando: {flags_ausentes}")
else:
    print("\nTodas as 14 flags confirmadas na base exportada.")

Flags esperadas:  14
Flags presentes:  14
Flags ausentes:   0

Todas as 14 flags confirmadas na base exportada.


In [35]:
# Storytelling e documentação executiva do tratamento dos dados

print("""
-------------------------------------------------------------------------
              RELATÓRIO DE LIMPEZA E TRATAMENTO DE DADOS
              Sistema Comercial de Saneamento — Micromedição
-------------------------------------------------------------------------

BASE ORIGINAL
-------------------------------------------------------------------------
  Registros totais:          1.912 ligações
  Colunas originais:           132 (15 cadastrais + 117 históricas)
  Períodos na base:             13 (mês base + _01 a _12)

PREMISSAS ADOTADAS
-------------------------------------------------------------------------
  - Colunas _01 a _12 compõem o histórico dos 12 meses conforme o case.
  - O mês base (sem sufixo) refere-se ao período atual de referência,
    usado para comparações e KPIs — não compõe o histórico.
  - VOLUME_LIDO >= 900.000 m³ é tratado como código de ocorrência de leitura.
  - VOLUME_REAL é considerado o dado confiável de consumo.
  - Nulos em economias e descontos são interpretados como ausência legítima (convertidos para 0).
  - Valores negativos em VALOR_SERVICOS representam estornos/créditos reais.
  - Outlier da matrícula 1371190-3 foi mantido no VOLUME_REAL, pois
    o sistema confirmou o estorno correspondente de R$ 280.240,43.

ETAPAS DE TRATAMENTO
-------------------------------------------------------------------------
  1. Padronização:
     - Normalização de strings e conversão de datas (formato dayfirst=True).
     - 82 datas nulas identificadas como esperadas (ligações sem hidrômetro).

  2. Tratamento de Nulos:
     - Economias e descontos convertidos para 0 (ausência legítima de categoria).
     - Demais casos sinalizados via flags de rastreabilidade.

  3. Tratamento de Volumes:
     - 107 leituras absurdas identificadas em 87 ligações (4,6% da base).
     - Pico de ocorrências em outubro (_10) com 35 casos.
     - Valores substituídos por NaN no VOLUME_LIDO para não distorcer médias.
     - VOLUME_REAL preservado intacto como referência de consumo efetivo.

  4. Inconsistências Financeiras:
     - 128 ligações com estornos em VALOR_SERVICOS (valores negativos).
     - 17 ligações com faturamento zerado e consumo > 0 no mês base.
       - 9 zeradas por estorno aplicado     -> R$ 15.022,62 a recuperar
       - 8 zeradas sem justificativa        -> R$  1.878,71 a recuperar
     - Potencial de recuperação no mês base: R$ 16.901,33.
     - Outlier extremo em agosto: estorno de R$ 280.240,43 (matrícula 1371190-3).

  5. Engenharia de Features (12 meses históricos — _01 a _12):
     - Métricas descritivas por ligação: média, total, min, max e desvio padrão.
     - Coeficiente de variação do consumo (CV) para detecção de instabilidade.
     - Contagem de meses com consumo zero por ligação.
     - Desvio do mês base frente à média histórica (absoluto e percentual).

DIAGNÓSTICO FINAL (FLAGS E SCORE DE RISCO)
-------------------------------------------------------------------------
  Total de flags geradas: 14

  Flags Cadastrais:
  - Situação água nula:          15 ligações  (0.8%)
  - Situação esgoto nula:        36 ligações  (1.9%)
  - Sem categoria:               17 ligações  (0.9%)
  - Sem hidrômetro (ativa):       1 ligação   (0.1%)
  - Sem economia preenchida:      6 ligações  (0.3%)

  Flags de Anomalia:
  - Leitura com ocorrência:      87 ligações  (4.6%)
  - Inconsistência água/esgoto:  65 ligações  (3.4%)
  - Serviço negativo:           128 ligações  (6.7%)
  - Faturamento zerado:          17 ligações  (0.9%)
  - Serviço extremo:              1 ligação   (0.1%)

  Flags Comportamentais:
  - Consumo zero recorrente:    124 ligações  (6.5%)
  - Alta variabilidade (CV>1.5): 89 ligações  (4.7%)
  - Todos os meses zerados:      10 ligações  (0.5%)
  - Subfaturamento:               0 ligações  (0.0%)
  - Hidrômetro antigo:            0 ligações  (0.0%)

  SCORE DE RISCO CONSOLIDADO:
  - Alto risco  (score >= 3):    27 ligações  (1.4%)
  - Risco médio (score >= 2):   120 ligações  (6.3%)
  - Sem anomalia (score = 0): 1.552 ligações (81.2%)

BASE FINAL EXPORTADA
-------------------------------------------------------------------------
  Colunas após tratamento: 161
  Arquivos gerados:
  - base_completa_tratada.xlsx   (1.912 registros)
  - base_ativas_tratada.xlsx     (1.815 registros)
  - base_nao_ativas_tratada.xlsx (   97 registros)
""")


-------------------------------------------------------------------------
              RELATÓRIO DE LIMPEZA E TRATAMENTO DE DADOS
              Sistema Comercial de Saneamento — Micromedição
-------------------------------------------------------------------------

BASE ORIGINAL
-------------------------------------------------------------------------
  Registros totais:          1.912 ligações
  Colunas originais:           132 (15 cadastrais + 117 históricas)
  Períodos na base:             13 (mês base + _01 a _12)

PREMISSAS ADOTADAS
-------------------------------------------------------------------------
  - Colunas _01 a _12 compõem o histórico dos 12 meses conforme o case.
  - O mês base (sem sufixo) refere-se ao período atual de referência,
    usado para comparações e KPIs — não compõe o histórico.
  - VOLUME_LIDO >= 900.000 m³ é tratado como código de ocorrência de leitura.
  - VOLUME_REAL é considerado o dado confiável de consumo.
  - Nulos em economias e descontos são in